# 14b: Statistical Analysis of Clinical Imaging Features vs Rejection

Source data: `../data/bd_estudiUPF.csv` (original clinical spreadsheet)

This notebook analyzes the ARFI elastography and DCE-US (contrast-enhanced ultrasound)
parameters that were manually measured by the radiologists. These are the same features
that Clara's paper (Bassaganyas et al. 2025) analyzed.

Features:
- ARFI: mediana, media, DE, RIQ (elastography/stiffness)
- DCE-US: PE, WiAUC, RT, mTTI, TTP, WiR, WiPi, WoAUC, WiWoAUC, FT, WoR (perfusion)
- QOF, Area (quality and region size)

Goal: reproduce Clara's analysis and compare with our radiomics results.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import os

print("Imports OK")

Imports OK


In [2]:
# load clinical data directly from source
clinical = pd.read_csv(os.path.join("..", "data", "bd_estudiUPF.csv"))
clinical["id estudio"] = clinical["id estudio"].astype(str).str.strip()
print(f"Clinical data: {clinical.shape[0]} rows, {clinical.shape[1]} columns")

# rename key columns for convenience
motivo_col = "motivo (1: 1 semana, 2: 1 mes, 3: 1 año, 4: sospecha, 5: seguimiento)="
clinical = clinical.rename(columns={
    "id estudio": "study_id",
    motivo_col: "motivo",
    "RECHAZO CLÍNICO": "rejection",
})

print(f"Rejection distribution:")
print(clinical["rejection"].value_counts().to_string())

Clinical data: 138 rows, 57 columns
Rejection distribution:
rejection
0    98
1    40


In [3]:
# define the clinical imaging features to test
# ARFI elastography (stiffness)
arfi_features = ["ARFI mediana", "ARFI media", "ARFI DE", "ARFI RIQ"]

# DCE-US perfusion parameters
dceus_features = ["PE", "WiAUC", "RT", "mTTI", "TTP", "WiR", "WiPi",
                  "WoAUC", "WiWoAUC", "FT", "WoR"]

# other
other_features = ["QOF", "Area"]

all_clinical_features = arfi_features + dceus_features + other_features

# check availability and non-null counts
print("Feature availability:")
for feat in all_clinical_features:
    if feat in clinical.columns:
        n_valid = clinical[feat].dropna().shape[0]
        print(f"  {feat}: {n_valid}/{len(clinical)} non-null")
    else:
        print(f"  {feat}: NOT FOUND")

Feature availability:
  ARFI mediana: 121/138 non-null
  ARFI media: 121/138 non-null
  ARFI DE: 121/138 non-null
  ARFI RIQ: 121/138 non-null
  PE: 127/138 non-null
  WiAUC: 127/138 non-null
  RT: 127/138 non-null
  mTTI: 127/138 non-null
  TTP: 127/138 non-null
  WiR: 127/138 non-null
  WiPi: 127/138 non-null
  WoAUC: 127/138 non-null
  WiWoAUC: 127/138 non-null
  FT: 127/138 non-null
  WoR: 127/138 non-null
  QOF: 127/138 non-null
  Area: 127/138 non-null


In [4]:
# split into groups
rejection = clinical[clinical["rejection"] == 1]
no_rejection = clinical[clinical["rejection"] == 0]

print(f"No rejection: {len(no_rejection)}")
print(f"Rejection: {len(rejection)}")

No rejection: 98
Rejection: 40


## Note on repeated measures

The 138 studies come from 56 patients (mean 2.5 studies per patient). Some patients
have multiple studies across different timepoints, and 14 patients have both rejection
and no-rejection studies.

All statistical tests below treat each study as an independent observation. This
matches the approach in Bassaganyas et al. 2025 and is standard for this type of
clinical imaging study. However, studies from the same patient are not truly
independent (they share genetics, surgical technique, baseline organ characteristics),
which can inflate the effective sample size.

See `docs/PLAN_REPEATED_MEASURES.md` for planned investigation of this limitation.

## Univariate testing -- full dataset
Using Mann-Whitney U (consistent with Clara's paper).

In [5]:
test_results = []

for feat in all_clinical_features:
    if feat not in clinical.columns:
        continue

    vals_no_rej = no_rejection[feat].dropna().values
    vals_rej = rejection[feat].dropna().values

    if len(vals_no_rej) < 3 or len(vals_rej) < 3:
        print(f"  Skipping {feat}: too few values")
        continue

    stat, p_value = stats.mannwhitneyu(vals_no_rej, vals_rej, alternative="two-sided")

    # rank-biserial correlation as effect size
    n1 = len(vals_no_rej)
    n2 = len(vals_rej)
    effect_size = 1 - (2 * stat) / (n1 * n2)

    test_results.append({
        "feature": feat,
        "n_no_rej": n1,
        "n_rej": n2,
        "statistic": stat,
        "p_value": p_value,
        "effect_size": effect_size,
        "median_no_rej": np.median(vals_no_rej),
        "median_rej": np.median(vals_rej),
    })

results_df = pd.DataFrame(test_results).sort_values("p_value").reset_index(drop=True)
print(f"Tests run: {len(results_df)}")

Tests run: 17


In [6]:
# show all results
significant = results_df[results_df["p_value"] < 0.05]
print(f"Features with p < 0.05: {len(significant)} out of {len(results_df)}")
print()

display_cols = ["feature", "p_value", "effect_size", "median_no_rej", "median_rej", "n_no_rej", "n_rej"]
print("All features (sorted by p-value):")
print(results_df[display_cols].to_string(index=False))

Features with p < 0.05: 2 out of 17

All features (sorted by p-value):
     feature  p_value  effect_size  median_no_rej  median_rej  n_no_rej  n_rej
ARFI mediana 0.027801     0.260675          1.255       1.470        88     33
  ARFI media 0.029492     0.257920          1.275       1.450        88     33
        Area 0.086323    -0.194444          0.460       0.460        91     36
         WiR 0.104467    -0.185592        170.680     115.255        91     36
        WiPi 0.121458    -0.177045        562.630     320.850        91     36
          PE 0.133475    -0.171551        881.490     512.065        91     36
       WiAUC 0.147885    -0.165446       3433.720    2286.650        91     36
     ARFI DE 0.159406     0.166667          0.190       0.230        88     33
     WiWoAUC 0.160246    -0.160562       9738.400    6201.640        91     36
       WoAUC 0.196390    -0.147741       6042.040    4253.920        91     36
    ARFI RIQ 0.243063     0.138430          0.270       0.30

## Stratified by time period
Clara found that ARFI and DCE-US parameters are only discriminative after 90 days post-transplant.

In [7]:
def run_mannwhitney_on_subset(subset_df, label):
    """Run Mann-Whitney on clinical features for a subset."""
    rej = subset_df[subset_df["rejection"] == 1]
    no_rej = subset_df[subset_df["rejection"] == 0]

    print(f"\n--- {label} ---")
    print(f"No rejection: {len(no_rej)}, Rejection: {len(rej)}")

    if len(rej) < 3 or len(no_rej) < 3:
        print("Too few samples in one group. Skipping.")
        return None

    rows = []
    for feat in all_clinical_features:
        if feat not in subset_df.columns:
            continue
        vals_no = no_rej[feat].dropna().values
        vals_yes = rej[feat].dropna().values
        if len(vals_no) < 3 or len(vals_yes) < 3:
            continue
        stat, p = stats.mannwhitneyu(vals_no, vals_yes, alternative="two-sided")
        effect = 1 - (2 * stat) / (len(vals_no) * len(vals_yes))
        rows.append({
            "feature": feat,
            "p_value": p,
            "effect_size": effect,
            "median_no_rej": np.median(vals_no),
            "median_rej": np.median(vals_yes),
        })

    sub_results = pd.DataFrame(rows).sort_values("p_value")
    sig = sub_results[sub_results["p_value"] < 0.05]
    print(f"Features with p < 0.05: {len(sig)}")
    print()
    print(sub_results[["feature", "p_value", "effect_size", "median_no_rej", "median_rej"]].to_string(index=False))

    return sub_results

early_results = run_mannwhitney_on_subset(
    clinical[clinical["motivo"].isin([1, 2])],
    "Early (motivo 1-2, within 90 days)")

late_results = run_mannwhitney_on_subset(
    clinical[clinical["motivo"].isin([3, 4, 5])],
    "Late (motivo 3-5, beyond 90 days)")


--- Early (motivo 1-2, within 90 days) ---
No rejection: 57, Rejection: 10
Features with p < 0.05: 0

     feature  p_value  effect_size  median_no_rej  median_rej
     WiWoAUC 0.108954     0.337374      10372.400   16745.210
       WiAUC 0.117750     0.329293       3487.230    6343.890
       WoAUC 0.137005     0.313131       6517.800   10401.330
          RT 0.205869     0.266667          5.050       7.070
     ARFI DE 0.231365    -0.267500          0.250       0.215
         TTP 0.275196     0.230303          8.820       9.220
         QOF 0.306050     0.216162         89.660      90.580
        Area 0.311250    -0.212121          0.510       0.460
          FT 0.315261     0.212121         10.150      12.250
        WiPi 0.334240     0.204040        639.870    1010.510
          PE 0.374347     0.187879       1012.470    1618.990
        mTTI 0.417296    -0.171717         28.800      21.940
    ARFI RIQ 0.572704    -0.127500          0.325       0.280
         WoR 0.670933     0.0

## Stratified by actual days post-transplant (replicating Clara's paper)

The analysis above used `motivo` as a proxy for time period. However, Clara's paper
splits by **actual days post-transplantation** using three groups: 0-10 days, 11-90
days, and > 90 days. The spreadsheet has a `Días pTXP` column with the exact number
of days. This section uses that column to replicate Clara's time-based split exactly.

In [8]:
# load the days post-transplant column
days_col = "Días pTXP"
clinical_raw = pd.read_csv(os.path.join("..", "data", "bd_estudiUPF.csv"))
clinical_raw["id estudio"] = clinical_raw["id estudio"].astype(str).str.strip()

# add days column to our working dataframe
clinical["days_post_tx"] = clinical_raw[days_col].values

print(f"Days post-transplant: {clinical['days_post_tx'].notna().sum()}/{len(clinical)} non-null")
print(f"Range: {clinical['days_post_tx'].min():.0f} to {clinical['days_post_tx'].max():.0f} days")
print()

# show how motivo maps to actual days
print("Median days by motivo:")
for m in sorted(clinical["motivo"].unique()):
    subset = clinical[clinical["motivo"] == m]["days_post_tx"]
    print(f"  motivo {m}: median={subset.median():.0f}, range=[{subset.min():.0f}-{subset.max():.0f}], n={len(subset)}")

print()

# count studies where motivo and actual days disagree on the 90-day split
motivo_early = clinical["motivo"].isin([1, 2])
days_early = clinical["days_post_tx"] <= 90
disagree = motivo_early != days_early
print(f"Studies where motivo-based and days-based splits disagree: {disagree.sum()}")
if disagree.sum() > 0:
    print(clinical[disagree][["study_id", "motivo", "days_post_tx", "rejection"]].to_string(index=False))

Days post-transplant: 138/138 non-null
Range: 3 to 8429 days

Median days by motivo:
  motivo 1: median=8, range=[3-17], n=30
  motivo 2: median=27, range=[12-90], n=37
  motivo 3: median=384, range=[358-462], n=27
  motivo 4: median=679, range=[11-8429], n=24
  motivo 5: median=314, range=[36-3535], n=20

Studies where motivo-based and days-based splits disagree: 13
study_id  motivo  days_post_tx  rejection
   01_03       5            45          1
   06_03       5            84          0
   08_01       5            36          1
   09_01       4            12          1
   09_02       4            17          1
   09_03       5            61          0
   14_03       4            61          1
   19_01       4            11          1
   19_02       4            20          0
   29_03       5            64          1
   36_02       5            82          0
   40_01       4            60          1
   43_02       4            15          0


In [9]:
# run the same analysis using Clara's exact time split: > 90 days
# this matches Table 2 and Table 3 in Bassaganyas et al. 2025

late_days = clinical[clinical["days_post_tx"] > 90]
late_days_results = run_mannwhitney_on_subset(late_days, "Late (> 90 days post-transplant, per Clara's paper)")

# also run the early periods for completeness
very_early = clinical[clinical["days_post_tx"] <= 10]
early_days = clinical[(clinical["days_post_tx"] > 10) & (clinical["days_post_tx"] <= 90)]

_ = run_mannwhitney_on_subset(very_early, "Very early (0-10 days)")
_ = run_mannwhitney_on_subset(early_days, "Early (11-90 days)")


--- Late (> 90 days post-transplant, per Clara's paper) ---
No rejection: 36, Rejection: 22
Features with p < 0.05: 8

     feature  p_value  effect_size  median_no_rej  median_rej
  ARFI media 0.000011     0.740032           0.97        1.44
ARFI mediana 0.000018     0.720893           0.97        1.46
     ARFI DE 0.000631     0.574163           0.15        0.23
    ARFI RIQ 0.003502     0.491228           0.23        0.29
       WiAUC 0.007854    -0.453311        3386.60     1589.36
       WoAUC 0.019362    -0.398981        5788.51     2827.77
     WiWoAUC 0.020423    -0.395586        9137.69     4627.94
        WiPi 0.043520    -0.344652         442.45      273.89
          PE 0.057595    -0.324278         656.08      439.15
         WiR 0.133867    -0.256367         126.04      109.56
          RT 0.141811    -0.251273           7.00        5.94
        mTTI 0.180528    -0.229202          29.14       26.24
         TTP 0.222743    -0.208829          11.50       10.34
          FT

## Comparison: motivo-based vs days-based stratification (> 90 days)

This table compares our motivo-based late subset with the days-based split and
Clara's published p-values, so we can see if using actual days aligns us closer
to the paper.

In [10]:
# compare the two stratification approaches with Clara's published values
# Clara's paper Table 2 and Table 3 p-values for > 90 days:
clara_pvalues = {
    "ARFI mediana": 0.001,  # reported as "<0.001"
    "WiAUC": 0.008,
    "WiPi": 0.044,
    "WoAUC": 0.019,
    "WiWoAUC": 0.020,
    "PE": 0.058,
    "WiR": 0.134,
    "RT": 0.276,
    "mTTI": 0.181,
    "TTP": 0.223,
    "FT": 0.238,
    "WoR": 0.617,
}

# build comparison table
if late_results is not None and late_days_results is not None:
    comparison_rows = []
    for feat in clara_pvalues:
        p_clara = clara_pvalues[feat]

        # motivo-based
        motivo_row = late_results[late_results["feature"] == feat]
        p_motivo = motivo_row["p_value"].values[0] if len(motivo_row) > 0 else None

        # days-based
        days_row = late_days_results[late_days_results["feature"] == feat]
        p_days = days_row["p_value"].values[0] if len(days_row) > 0 else None

        comparison_rows.append({
            "feature": feat,
            "p_clara_paper": f"{p_clara:.3f}" if p_clara >= 0.001 else "<0.001",
            "p_motivo_split": f"{p_motivo:.3f}" if p_motivo is not None else "N/A",
            "p_days_split": f"{p_days:.3f}" if p_days is not None else "N/A",
        })

    comp_df = pd.DataFrame(comparison_rows)
    print("Comparison of p-values: Clara's paper vs our motivo-based vs our days-based split")
    print()
    print(comp_df.to_string(index=False))
    print()
    print("If the days-based split is closer to Clara's values, it confirms that")
    print("the motivo proxy was the source of discrepancy identified in the comparison report.")

Comparison of p-values: Clara's paper vs our motivo-based vs our days-based split

     feature p_clara_paper p_motivo_split p_days_split
ARFI mediana         0.001          0.000        0.000
       WiAUC         0.008          0.018        0.008
        WiPi         0.044          0.066        0.044
       WoAUC         0.019          0.046        0.019
     WiWoAUC         0.020          0.043        0.020
          PE         0.058          0.081        0.058
         WiR         0.134          0.143        0.134
          RT         0.276          0.397        0.142
        mTTI         0.181          0.546        0.181
         TTP         0.223          0.314        0.223
          FT         0.238          0.652        0.238
         WoR         0.617          0.492        0.617

If the days-based split is closer to Clara's values, it confirms that
the motivo proxy was the source of discrepancy identified in the comparison report.


In [11]:
# save
output_path = os.path.join("reports", "14b_stats_clinical_features.csv")
results_df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

print()
print("SUMMARY")
print(f"Total clinical features tested: {len(results_df)}")
print(f"Uncorrected p < 0.05: {len(results_df[results_df['p_value'] < 0.05])}")

Saved to reports/14b_stats_clinical_features.csv

SUMMARY
Total clinical features tested: 17
Uncorrected p < 0.05: 2
